# 05 — Semantic Chunking + Embedding Smoke Test

**Phase 5** (plan §6 Stage 5). Runs after `04_layout_analysis.ipynb`.

## What this notebook does

1. **Pre-flight check** — verifies Neo4j connectivity, confirms fusion +
   layout are ready, checks CHUNK vector index exists.
2. **Chunking smoke test** — runs `chunk_pages()` on a 50-page sample
   (both `markdown_section` and `sliding_window` strategies exercised)
   and inspects the resulting `CHUNK` nodes.
3. **Embedding smoke test** — embeds 20 chunks via Silra `text-embedding-v4`
   (1 024-dim) and verifies the vectors are correct shape in Neo4j.
4. **Strategy bake-off summary** — compares chunk size distribution and
   strategy breakdown so you can tune before the full-corpus run.
5. **Artefact write** — saves `notebooks/_artifacts/05_chunking/chunking.json`.

## Production background runners

| Step | Script | Est. runtime |
|---|---|---|
| Full-corpus fusion | `caffeinate -dimsu uv run python scripts/run_fusion.py` | 30–60 min |
| Full-corpus chunking | `caffeinate -dimsu uv run python scripts/run_chunking.py` | 15–30 min |
| Full-corpus embedding | `caffeinate -dimsu uv run python scripts/run_embedding.py` | ~55 min |

Run those **after** this notebook confirms the pipeline is correct.

## Text source priority per page

| Priority | Source | Strategy |
|---|---|---|
| 1 | `structuredMarkdown` (`layoutStatus='ok'`) | `markdown_section` |
| 2 | `textFused` (`fusionStatus in ['ok','single']`) | `sliding_window` |
| 3 | `text` (native-text pages) | `sliding_window` |

**Next**: Phase 6 — `06_kg_construction.ipynb` (keyword extraction → KEYWORD nodes + MENTION edges).

In [5]:
import json
import logging
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

# --- repo root on sys.path so we can import apps.backend.*
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "apps").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
)
logging.getLogger("neo4j.notifications").setLevel(logging.WARNING)

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "_artifacts" / "05_chunking"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# --- smoke-run config (flip RUN_FULL=True or set env RUN_FULL=1 for full corpus)
RUN_FULL   = os.environ.get("RUN_FULL", "").lower() in ("1", "true", "yes")
MAX_PAGES  = None if RUN_FULL else int(os.environ.get("MAX_PAGES", "50"))
EMBED_SAMPLE = 20  # chunks to embed in the smoke test
CHUNK_SIZE = 500
OVERLAP    = 50

print(f"mode     : {'FULL CORPUS' if RUN_FULL else f'SMOKE (max_pages={MAX_PAGES})'}")
print(f"chunk_size={CHUNK_SIZE}  overlap={OVERLAP}  embed_sample={EMBED_SAMPLE}")
print(f"artifact : {ARTIFACT_DIR}")

mode     : SMOKE (max_pages=50)
chunk_size=500  overlap=50  embed_sample=20
artifact : /Users/mohasani/Ancient/notebooks/_artifacts/05_chunking


---
## 1 — Pre-flight: connectivity + pipeline readiness

In [6]:
from neo4j import GraphDatabase

# Load prior artefact for chain validation
layout_art = REPO_ROOT / "notebooks" / "_artifacts" / "04_layout_analysis" / "report.json"
if layout_art.exists():
    prev = json.loads(layout_art.read_text())
    print(f"[04_layout_analysis] pages_ok={prev.get('pages_ok')} pages_total={prev.get('pages_total')}")
else:
    print("[04_layout_analysis] artifact missing — layout notebook not yet run")

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    auth=(os.getenv("NEO4J_USERNAME", "neo4j"), os.getenv("NEO4J_PASSWORD", "AncientChina")),
    notifications_disabled_classifications=["UNRECOGNIZED"],
)

with driver.session() as s:
    totals = s.run("""
        MATCH (p:PAGE)
        RETURN
          count(p)                                                         AS total_pages,
          count(CASE WHEN p.text IS NOT NULL AND p.mode <> 'ocr' THEN 1 END)
                                                                           AS native_pages,
          count(CASE WHEN p.fusionStatus IN ['ok','single'] THEN 1 END)   AS fused_pages,
          count(CASE WHEN p.layoutStatus = 'ok'            THEN 1 END)   AS layout_ok,
          count(CASE WHEN p.chunkingAt IS NOT NULL         THEN 1 END)   AS already_chunked
    """).single()
    t = dict(totals)

print(f"\nNeo4j PAGE summary:")
for k, v in t.items():
    print(f"  {k:25s} = {v}")

chunkable = t["native_pages"] + t["fused_pages"]
print(f"\nEstimated chunkable pages (native + fused) = {chunkable:,}")
if chunkable == 0:
    raise RuntimeError(
        "No chunkable pages found. "
        "Run scripts/run_fusion.py first so fused_pages > 0."
    )

[04_layout_analysis] pages_ok=20 pages_total=20

Neo4j PAGE summary:
  total_pages               = 19221
  native_pages              = 11745
  fused_pages               = 2886
  layout_ok                 = 2820
  already_chunked           = 7040

Estimated chunkable pages (native + fused) = 14,631


In [8]:
# Verify the chunk_embedding_vector_index exists (or can be created)
with driver.session() as s:
    indexes = s.run("""
        SHOW INDEXES
        YIELD name, type, state
        WHERE name STARTS WITH 'chunk'
        RETURN name, type, state
        ORDER BY name
    """).data()

print("CHUNK-related indexes:")
for idx in indexes:
    print(f"  {idx['name']:45s} {idx['type']:12s} {idx['state']}")

vector_idx_names = {i["name"] for i in indexes if i["type"] == "VECTOR"}
print(f"\nVector indexes: {vector_idx_names or '(none — will be created by init_schema)'}")

if not vector_idx_names:
    print("\nCreating schema (vector indexes + constraints)...")
    from apps.backend.graph.schema import init_schema
    init_schema(driver)
    print("Schema initialised.")

CHUNK-related indexes:
  chunk_editorial_layer_index                   RANGE        ONLINE
  chunk_embedding_classical                     VECTOR       ONLINE
  chunk_embedding_vernacular                    VECTOR       ONLINE
  chunk_id_unique                               RANGE        ONLINE
  chunk_language_index                          RANGE        ONLINE
  chunk_section_index                           RANGE        ONLINE
  chunk_tier_index                              RANGE        ONLINE

Vector indexes: {'chunk_embedding_vernacular', 'chunk_embedding_classical'}


---
## 2 — Chunking smoke test (50 pages, both strategies)

In [9]:
from apps.backend.pipeline.chunk import chunk_pages, resolve_page_text

print(f"Running chunk_pages(max_pages={MAX_PAGES}, chunk_size={CHUNK_SIZE}, overlap={OVERLAP})...")
chunk_report = chunk_pages(
    driver,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
    max_pages=MAX_PAGES,
    recompute=False,
)

print(f"\nChunking report:")
for k, v in chunk_report.to_dict().items():
    if k != "errors":
        print(f"  {k:30s} = {v}")
if chunk_report.errors:
    print(f"  errors (first 5): {chunk_report.errors[:5]}")

Running chunk_pages(max_pages=50, chunk_size=500, overlap=50)...


2026-05-26 16:37:56,136 INFO     apps.backend.pipeline.chunk: Batch skip=0 rows=200 chunks_this_batch=200 total_chunks=200



Chunking report:
  pages_total                    = 200
  pages_chunked                  = 162
  pages_skipped                  = 38
  pages_failed                   = 0
  chunks_created                 = 200
  duration_seconds               = 0.2310628890991211


In [10]:
# Inspect CHUNK distribution in Neo4j
with driver.session() as s:
    stats = s.run("""
        MATCH (c:CHUNK)
        RETURN
          count(c)                                                                  AS total_chunks,
          avg(c.charCount)                                                          AS avg_chars,
          min(c.charCount)                                                          AS min_chars,
          max(c.charCount)                                                          AS max_chars,
          count(CASE WHEN c.chunkStrategy = 'markdown_section' THEN 1 END)         AS md_chunks,
          count(CASE WHEN c.chunkStrategy = 'sliding_window'   THEN 1 END)         AS sw_chunks,
          count(CASE WHEN c.embeddingStatus = 'pending'        THEN 1 END)         AS pending_embed,
          count(DISTINCT c.documentId)                                              AS docs_covered
    """).single()
    cs = dict(stats)

print("CHUNK node stats:")
for k, v in cs.items():
    vfmt = f"{v:.1f}" if isinstance(v, float) else str(v)
    print(f"  {k:30s} = {vfmt}")

assert cs["total_chunks"] > 0, "No chunks written — check page text availability"
strategy_dist = {"markdown_section": cs["md_chunks"], "sliding_window": cs["sw_chunks"]}
print(f"\nStrategy mix: {strategy_dist}")

CHUNK node stats:
  total_chunks                   = 21960
  avg_chars                      = 420.2
  min_chars                      = 10
  max_chars                      = 500
  md_chunks                      = 1887
  sw_chunks                      = 20073
  pending_embed                  = 11080
  docs_covered                   = 42

Strategy mix: {'markdown_section': 1887, 'sliding_window': 20073}


In [11]:
# Fetch sample chunks for inspection
with driver.session() as s:
    samples = s.run("""
        MATCH (c:CHUNK)
        RETURN c.id AS chunk_id,
               c.documentId AS doc,
               c.chunkStrategy AS strategy,
               c.charCount AS chars,
               c.language AS lang,
               left(c.text, 120) AS preview
        ORDER BY c.id
        LIMIT 6
    """).data()

print("Sample chunks:")
for i, row in enumerate(samples, 1):
    print(f"\n[{i}] {row['chunk_id']}")
    print(f"    doc      : {row['doc']}")
    print(f"    strategy : {row['strategy']}  chars={row['chars']}  lang={row['lang']}")
    print(f"    preview  : {row['preview']}...")

Sample chunks:

[1] 刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00002::chunk_0000
    doc      : 刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df
    strategy : markdown_section  chars=171  lang=zh-classical
    preview  : 中华書局出  版 （北京王府井大街36號） 新华書店北京發行所發行 北京冠中印刷廠印刷 * 850×1168毫米1/32·19印張·1插页·407千字 1989年 3 月第 1 版 1989年3 月北京第1次印刷 印数1—2000衍定领：8...

[2] 刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00003::chunk_0000
    doc      : 刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df
    strategy : sliding_window  chars=310  lang=zh-classical
    preview  : K)1+1-
十和卜计产动法
严事宁立个之中出中叫
应本编心的
本计为而子的时以话动子种对部甘
子四冲时梁冲子时如的文严并中
时则高再
计立的制等水卜湖年的可前饼亡叫对
再
十文计始/多水6水卜产入曲空正反
A卜在时画节1产
的藏河字十七就...

[3] 刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00004::chunk_0000
    doc      : 刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df
    strategy : sliding_window  chars=292  lang=zh-modern
    preview  : 分乡文〉的溶为主关一生英高
7国声今
并带实工子安分四的之画沟户酬游
本*号二心油出
布关决3一
合离节广四和间单
角
山分工子A方中治0A商广
与下品5分和陪出三就兴施和部中作含高
7齐*上说二子车部取集m
页常粉成P,2507阳
安爸产...

[4] 刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00005::chunk_0000

---
## 3 — Strategy comparison: markdown_section vs sliding_window

In [12]:
with driver.session() as s:
    per_strategy = s.run("""
        MATCH (c:CHUNK)
        RETURN c.chunkStrategy AS strategy,
               count(c)        AS n,
               avg(c.charCount) AS avg_chars,
               min(c.charCount) AS min_chars,
               max(c.charCount) AS max_chars
        ORDER BY strategy
    """).data()

print(f"{'Strategy':<20} {'N':>8} {'Avg chars':>12} {'Min':>8} {'Max':>8}")
print("-" * 60)
for r in per_strategy:
    avg = f"{r['avg_chars']:.1f}" if r['avg_chars'] else "—"
    print(f"{r['strategy']:<20} {r['n']:>8,} {avg:>12} {r['min_chars']:>8} {r['max_chars']:>8}")

# Show a side-by-side example from the same page if possible
with driver.session() as s:
    md_example = s.run("""
        MATCH (c:CHUNK)
        WHERE c.chunkStrategy = 'markdown_section'
        RETURN c.id AS id, c.charCount AS chars, left(c.text, 200) AS text
        LIMIT 1
    """).data()
    sw_example = s.run("""
        MATCH (c:CHUNK)
        WHERE c.chunkStrategy = 'sliding_window'
        RETURN c.id AS id, c.charCount AS chars, left(c.text, 200) AS text
        LIMIT 1
    """).data()

if md_example:
    r = md_example[0]
    print(f"\n--- markdown_section example (chars={r['chars']}) ---")
    print(r["text"])

if sw_example:
    r = sw_example[0]
    print(f"\n--- sliding_window example (chars={r['chars']}) ---")
    print(r["text"])

Strategy                    N    Avg chars      Min      Max
------------------------------------------------------------
markdown_section        1,887        348.2       10      500
sliding_window         20,073        427.0       19      500

--- markdown_section example (chars=171) ---
中华書局出  版 （北京王府井大街36號） 新华書店北京發行所發行 北京冠中印刷廠印刷 * 850×1168毫米1/32·19印張·1插页·407千字 1989年 3 月第 1 版 1989年3 月北京第1次印刷 印数1—2000衍定领：8.10元 ISBN 7—101—00472--5/K·208


# 敦煌吐香番唐代法制文考 劉俊文著

--- sliding_window example (chars=310) ---
K)1+1-
十和卜计产动法
严事宁立个之中出中叫
应本编心的
本计为而子的时以话动子种对部甘
子四冲时梁冲子时如的文严并中
时则高再
计立的制等水卜湖年的可前饼亡叫对
再
十文计始/多水6水卜产入曲空正反
A卜在时画节1产
的藏河字十七就阴元性货
础算水文文十卜π心中1品宝确节亡
部计前法
时可曾文站站六
*中冯下本时少产少定
实为京实兴分之帝头管才动事四
州个时完出中方
空为中
州中钟川部八时


---
## 4 — Embedding smoke test (20 chunks via Silra text-embedding-v4)

In [ ]:
import importlib
import apps.backend.pipeline.embed as _embed_mod
importlib.reload(_embed_mod)
from apps.backend.pipeline.embed import embed_chunks

_SILRA_BATCH_LIMIT = 10  # Silra text-embedding-v4 hard cap — confirmed 2026-05-26

embed_model = os.getenv("EMBED_LLM_MODEL", "text-embedding-v4")
print(f"Embedding model : {embed_model}")
print(f"Embedding dims  : 1024 (canonical — AGENTS.md 2026-05-16)")
print(f"Batch size      : {_SILRA_BATCH_LIMIT} (Silra hard limit — AGENTS.md 2026-05-26)")
print(f"Sample size     : {EMBED_SAMPLE} chunks\n")

embed_report = embed_chunks(
    driver,
    model=embed_model,
    batch_size=_SILRA_BATCH_LIMIT,
    max_chunks=EMBED_SAMPLE,
    recompute=False,
)

print("Embedding report:")
for k, v in embed_report.to_dict().items():
    if k != "errors":
        print(f"  {k:30s} = {v}")
if embed_report.errors:
    print(f"  errors: {embed_report.errors}")

Embedding model : text-embedding-v4
Embedding dims  : 1024 (canonical — AGENTS.md 2026-05-16)
Batch size      : 10 (Silra hard limit — AGENTS.md 2026-05-26)
Sample size     : 20 chunks



2026-05-26 16:42:19,145 INFO     httpx: HTTP Request: POST https://api.silra.cn/v1/embeddings "HTTP/1.1 400 Bad Request"
2026-05-26 16:42:19,146 WARNING  apps.backend.pipeline.embed: embed batch attempt 1 failed (Error code: 400 - {'error': {'message': '<400> ***.***.InvalidParameter: Value error, batch size is invalid, it should not be larger than 10.: ***.contents', 'type': 'InvalidParameter', 'param': '', 'code': 'InvalidParameter'}}), retry in 1s
2026-05-26 16:42:26,349 INFO     httpx: HTTP Request: POST https://api.silra.cn/v1/embeddings "HTTP/1.1 400 Bad Request"
2026-05-26 16:42:26,351 WARNING  apps.backend.pipeline.embed: embed batch attempt 2 failed (Error code: 400 - {'error': {'message': '<400> ***.***.InvalidParameter: Value error, batch size is invalid, it should not be larger than 10.: ***.contents', 'type': 'InvalidParameter', 'param': '', 'code': 'InvalidParameter'}}), retry in 2s
2026-05-26 16:42:35,417 INFO     httpx: HTTP Request: POST https://api.silra.cn/v1/embeddi

In [ ]:
# Verify shape + value range of stored embeddings
with driver.session() as s:
    embed_stats = s.run("""
        MATCH (c:CHUNK)
        WHERE c.embeddingStatus = 'ok'
        RETURN c.id AS id,
               c.embeddingModel AS model,
               c.embeddingDims  AS dims,
               size(c.embedding) AS vec_len,
               c.embedding[0]   AS first_val
        LIMIT 4
    """).data()

print("Embedded chunks (sample):")
for row in embed_stats:
    print(f"  id={row['id'][:60]}...")
    print(f"    model={row['model']}  dims={row['dims']}  vec_len={row['vec_len']}  first={row['first_val']:.6f}")
    assert row["vec_len"] == 1024, f"Expected 1024-dim vector, got {row['vec_len']}"

print("\nAll sampled embeddings are 1024-dim. Vector index ready for Phase 7 similarity search.")

---
## 5 — Full-corpus production runs (background scripts)

The smoke test is passing. Run these sequentially in a terminal before starting Phase 6:

```bash
# Step 1 — full-corpus OCR fusion (~30–60 min, no API calls)
caffeinate -dimsu uv run python scripts/run_fusion.py \
    --log-file logs/fusion_run.log --verbose

# Step 2 — full-corpus chunking (~15–30 min)
caffeinate -dimsu uv run python scripts/run_chunking.py \
    --log-file logs/chunking_run.log --verbose

# Step 3 — full-corpus embedding (~55 min, Silra API)
caffeinate -dimsu uv run python scripts/run_embedding.py \
    --log-file logs/embedding_run.log --verbose
```

Monitor progress:
```bash
tail -f logs/fusion_run.log
tail -f logs/chunking_run.log
tail -f logs/embedding_run.log
```

In [ ]:
# Show final pipeline status
with driver.session() as s:
    pipeline_status = s.run("""
        MATCH (p:PAGE)
        RETURN
          count(p)                                                        AS total_pages,
          count(CASE WHEN p.text IS NOT NULL AND p.mode <> 'ocr' THEN 1 END) AS native_text,
          count(CASE WHEN p.fusionStatus IN ['ok','single'] THEN 1 END)  AS fused,
          count(CASE WHEN p.layoutStatus = 'ok'            THEN 1 END)  AS layout_ok,
          count(CASE WHEN p.chunkingAt IS NOT NULL         THEN 1 END)  AS chunked
    """).single()

    chunk_status = s.run("""
        MATCH (c:CHUNK)
        RETURN
          count(c)                                                              AS total_chunks,
          count(CASE WHEN c.embeddingStatus = 'ok'      THEN 1 END)            AS embedded,
          count(CASE WHEN c.embeddingStatus = 'pending' THEN 1 END)            AS pending,
          count(CASE WHEN c.embeddingStatus = 'failed'  THEN 1 END)            AS failed,
          count(CASE WHEN c.chunkStrategy = 'markdown_section' THEN 1 END)     AS md_chunks,
          count(CASE WHEN c.chunkStrategy = 'sliding_window'   THEN 1 END)     AS sw_chunks
    """).single()

ps = dict(pipeline_status)
cs2 = dict(chunk_status)

print("Pipeline status snapshot:")
print(f"  Pages total          : {ps['total_pages']:,}")
print(f"    native-text        : {ps['native_text']:,}")
print(f"    fused (OCR)        : {ps['fused']:,}")
print(f"    layout ok          : {ps['layout_ok']:,}")
print(f"    chunked            : {ps['chunked']:,}")
print()
print(f"  CHUNK nodes total    : {cs2['total_chunks']:,}")
print(f"    markdown_section   : {cs2['md_chunks']:,}")
print(f"    sliding_window     : {cs2['sw_chunks']:,}")
print(f"    embedded (ok)      : {cs2['embedded']:,}")
print(f"    pending embed      : {cs2['pending']:,}")
print(f"    failed embed       : {cs2['failed']:,}")

---
## 6 — Save artefact

In [ ]:
# Collect sample chunks for artefact
with driver.session() as s:
    sample_chunks = s.run("""
        MATCH (c:CHUNK)
        RETURN c.id AS chunk_id,
               c.chunkStrategy AS strategy,
               c.charCount AS chars,
               c.language AS lang,
               left(c.text, 100) AS text_preview
        ORDER BY c.id
        LIMIT 10
    """).data()

artifact = {
    "phase": "05_chunking_embeddings_bakeoff",
    "ts": datetime.now(timezone.utc).isoformat(),
    "config": {
        "chunk_size": CHUNK_SIZE,
        "overlap": OVERLAP,
        "max_pages": MAX_PAGES,
        "embed_sample": EMBED_SAMPLE,
        "run_full": RUN_FULL,
    },
    "pipeline_status": ps,
    "chunk_report": chunk_report.to_dict(),
    "embed_report": embed_report.to_dict(),
    "strategy_distribution": strategy_dist,
    "chunk_stats": cs,
    "total_chunks_in_graph": cs2["total_chunks"],
    "total_embedded": cs2["embedded"],
    "sample_chunks": [
        {
            "chunk_id": r["chunk_id"],
            "strategy": r["strategy"],
            "chars": r["chars"],
            "lang": r["lang"],
            "text": r["text_preview"],
        }
        for r in sample_chunks
    ],
}

artifact_path = ARTIFACT_DIR / "chunking.json"
artifact_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False))
print(f"Artefact written → {artifact_path}")
print(f"  chunk_report  : pages_chunked={chunk_report.pages_chunked}  chunks_created={chunk_report.chunks_created}")
print(f"  embed_report  : embedded={embed_report.chunks_embedded}  failed={embed_report.chunks_failed}")
print(f"  total CHUNK nodes in graph : {cs2['total_chunks']:,}")
print(f"  total embedded             : {cs2['embedded']:,}")
print("\nPhase 5 smoke test complete — ready for full-corpus background runs.")

In [ ]:
# Teardown — leave driver open for rest of kernel session (AGENTS.md rule 2026-05-18)
# driver.close()  ← DO NOT CALL

print("""
=== NEXT STEPS ===

1. Run full-corpus fusion (if not yet done):
   caffeinate -dimsu uv run python scripts/run_fusion.py --verbose

2. Run full-corpus chunking:
   caffeinate -dimsu uv run python scripts/run_chunking.py --verbose

3. Run full-corpus embedding (~55 min):
   caffeinate -dimsu uv run python scripts/run_embedding.py --verbose

4. Proceed to Phase 6: 06_kg_construction.ipynb
   (keyword extraction → KEYWORD nodes + (:CHUNK)-[:MENTION]->(:KEYWORD))
""")